# Bloco 3 — Notebook 01: Download Causal 1h, 4h e 1m (`2022-01-01`+)
Baixa o histórico público (`https://data-api.binance.vision`) das 7 moedas do universo (`BTCUSDT`, `ETHUSDT`, `SOLUSDT`, `XRPUSDT`, `ADAUSDT`, `LINKUSDT`, `AVAXUSDT`) via paginação automática (`backfill_historical_range`), preservando as colunas 0..10 (incluindo `quote_volume`, `number_of_trades`, `taker_buy_base_volume` e `taker_buy_quote_volume`), normaliza `close_time_ms` (`+1ms`) e executa o alinhamento causal `merge_asof` (`direction='backward'`, `allow_exact_matches=True`).


In [ ]:
# Bootstrap automático no Google Colab (clona ou atualiza o repositório e instala dependências)
import os, subprocess, sys
from pathlib import Path
REPO_URL = 'https://github.com/pac-coins/quant-crypto-chameleon-radar-engine.git'
REPO_DIR = Path('quant-crypto-chameleon-radar-engine')
if 'google.colab' in sys.modules:
    if not Path('config/strategies.yaml').exists():
        if not REPO_DIR.exists():
            subprocess.run(['git', 'clone', REPO_URL], check=True)
        os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pydantic', 'pydantic-settings', 'PyYAML', 'pandas', 'requests', 'cryptography'], check=True)
if '.' not in sys.path:
    sys.path.insert(0, '.')
# Garante variáveis mínimas caso não exista arquivo .env no ambiente do Colab
os.environ.setdefault('MODE', 'SHADOW_CHAMELEON')
os.environ.setdefault('TELEGRAM_BOT_TOKEN', 'colab_backtest_token_placeholder')
os.environ.setdefault('TELEGRAM_CHAT_ID', '0')


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from src.core.config import load_config
from src.core.database import Database
from src.core.binance_client import MonotonicClock, PublicBinanceClient, WeightGovernor
from src.data.candle_collector import CandleCollector, load_frame, align_btc_context
from src.features.feature_store import FeatureStore, FeatureRequest
from src.regimes.regime_detector import RegimeDetector
from src.strategies.bull_trend_pullback import BullTrendPullbackStrategy

cfg = load_config()
SYMBOLS = list(cfg.strategies.universe.symbols)
START_DATE_STR = cfg.strategies.universe.backtest_min_start_date  # '2022-01-01'
START_MS = int(datetime.fromisoformat(f'{START_DATE_STR}T00:00:00+00:00').timestamp() * 1000)
print('Universo de Execução (7 pares, sem BNBUSDT):', SYMBOLS, '| Início UTC:', START_DATE_STR, f'({START_MS})')


In [ ]:
# Inicializa SQLite WAL e Coletor Público (zero chaves privadas)
db_path = Path('data/colab_backtest.sqlite')
db_path.parent.mkdir(parents=True, exist_ok=True)
db = Database(db_path)
governor = WeightGovernor()
client = PublicBinanceClient(base_url='https://data-api.binance.vision', governor=governor)
clock = MonotonicClock(safety_margin_ms=cfg.strategies.execution_and_resilience.clock_safety_margin_ms)

detector = RegimeDetector(cfg.strategies.regimes_and_strategies.regime_thresholds_4h)
bull_strategy = BullTrendPullbackStrategy(enabled=True)
store = FeatureStore(
    consumers=[detector, bull_strategy],
    extra_requests=[
        FeatureRequest('ema', 9, '1h'),
        FeatureRequest('ema', 50, '1h'),
        FeatureRequest('ema', 200, '1h'),
        FeatureRequest('rvol', 20, '1h'),
    ],
)

def download_and_build_aligned_universe(start_ms: int = START_MS, end_ms: int | None = None):
    clock.sync(client.get_server_time_ms)
    collector = CandleCollector(client=client, db=db, clock=clock)
    for sym in SYMBOLS:
        n_1h = collector.backfill_historical_range(sym, '1h', start_time_ms=start_ms, end_time_ms=end_ms)
        n_4h = collector.backfill_historical_range(sym, '4h', start_time_ms=start_ms, end_time_ms=end_ms)
        print(f'[{sym}] Persistidos no SQLite WAL: 1h={n_1h} velas | 4h={n_4h} velas')
    df_btc_1h = load_frame(db, 'BTCUSDT', '1h', limit=100_000)
    aligned = {}
    for sym in SYMBOLS:
        df_1h = load_frame(db, sym, '1h', limit=100_000)
        df_4h = load_frame(db, sym, '4h', limit=30_000)
        built = store.build(df_1h, df_4h)
        aligned[sym] = align_btc_context(built, df_btc_1h[[c for c in ('close_time_ms', 'close') if c in df_btc_1h.columns]])
    return aligned
